# Digikala Recommendation Prediction — Final Evaluation

This notebook covers only the `Recommendation Prediction` component of final-system evaluation. The model is **not trained or tuned here**; the locked test split is reproduced, analyzed, and documented. The primary metric is `Macro-F1`, supplemented by uncertainty, slice, latency, cost, and failure analysis.


## Running on Kaggle

1. Create a fresh notebook with a T4 GPU and enable Internet.
2. Attach the Transformer notebook output with `Add Input`; `best_transformer_encoder`, `transformer_run_summary.json`, and the manifest are required.
3. Also attach the classical-baseline notebook output to enable a paired bootstrap comparison.
4. Run cells in order. This notebook performs no training and should finish much faster than the preceding notebook.
5. Never use test to change class weights, hyperparameters, or model selection. Later experiments require validation or a new holdout.


In [ ]:
from __future__ import annotations

import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48,<5',
    'sentencepiece>=0.2',
    'safetensors>=0.4',
])
print('Evaluation dependencies are ready.')


In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import socket
import time
import urllib.request
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
VALID_LABELS = ['recommended', 'not_recommended', 'no_idea']
LABEL2ID = {label: index for index, label in enumerate(VALID_LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}

HF_REPO_ID = 'RadeAI/Digikala_comments_products'
HF_REVISION = '89c3133b169c8d3793db8834f56f32fee33d9db0'
HF_FILENAME = 'digikala-comments.csv'
HF_EXPECTED_SIZE = 1_278_526_959
HF_EXPECTED_SHA256 = 'c7a8aa3020334fde8ec24944576a03fe5785e6fe12cd01042f5836632ddf8297'
HF_DOWNLOAD_URL = f'https://huggingface.co/datasets/{HF_REPO_ID}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true'

EXPECTED_TEST_ROWS = 9_662
EXPECTED_TEST_GROUPS = 8_215
EXPECTED_TEST_LABEL_COUNTS = {'recommended': 7_612, 'not_recommended': 980, 'no_idea': 1_070}
BASELINE_TEST = {
    'macro_f1': 0.6611412090831573,
    'weighted_f1': 0.8383702565246943,
    'accuracy': 0.8415441937487063,
    'f1_recommended': 0.9234181343770385,
    'f1_not_recommended': 0.6992366412213741,
    'f1_no_idea': 0.36076885165105965,
}
MIN_TEST_MACRO_GAIN = 0.02
MAX_PER_CLASS_F1_REGRESSION = 0.02
MAX_REPRODUCTION_MACRO_F1_DELTA = 0.001
P95_SINGLE_LATENCY_BUDGET_MS = 250.0
BOOTSTRAP_ITERATIONS = 1_000
PREDICT_BATCH_SIZE = 16
LATENCY_SINGLE_SAMPLES = 200
LATENCY_BATCH_SIZE = 32
LATENCY_BATCH_SAMPLES = 1_024
CHUNK_SIZE = 250_000

MODEL_DIR_OVERRIDE = os.getenv('DIGIKALA_TRANSFORMER_MODEL_DIR') or None
TRANSFORMER_SUMMARY_OVERRIDE = os.getenv('DIGIKALA_TRANSFORMER_SUMMARY') or None
MANIFEST_OVERRIDE = os.getenv('DIGIKALA_MANIFEST_PATH') or None
BASELINE_MODEL_OVERRIDE = os.getenv('DIGIKALA_BASELINE_MODEL') or None
COMMENTS_PATH_OVERRIDE = os.getenv('DIGIKALA_COMMENTS_PATH') or None

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'outputs' / 'recommendation_evaluation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
if torch.cuda.is_available():
    capability = tuple(torch.cuda.get_device_capability(0))
    device_arch = f'sm_{capability[0]}{capability[1]}'
    if device_arch not in torch.cuda.get_arch_list():
        raise RuntimeError(f'PyTorch build does not support {device_arch}: {torch.cuda.get_arch_list()}')
else:
    capability = None

print(json.dumps({
    'device': str(device),
    'gpu': gpu_name,
    'gpu_compute_capability': capability,
    'torch': torch.__version__,
    'torch_cuda_runtime': torch.version.cuda,
    'transformers': transformers.__version__,
    'output_dir': str(OUTPUT_DIR),
}, ensure_ascii=False, indent=2))


## Discover and validate artifacts

Paths are discovered from Kaggle Input. If multiple similar runs are attached, set the overrides at the top of the notebook. The baseline artifact is optional but strongly recommended for paired bootstrap.


In [ ]:
SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]

def find_named_file(filename: str, override: str | None = None, required: bool = True) -> Path | None:
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    candidates = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(path for path in root.rglob(filename) if path.is_file())
    candidates = [path for path in candidates if OUTPUT_DIR not in path.parents]
    if not candidates:
        if required:
            raise FileNotFoundError(f'{filename} was not found. Attach the previous notebook output with Add Input.')
        return None
    return sorted(candidates, key=lambda path: len(str(path)))[0]

def find_named_directory(dirname: str, override: str | None = None) -> Path:
    if override:
        path = Path(override)
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    candidates = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(path for path in root.rglob(dirname) if path.is_dir())
    candidates = [path for path in candidates if (path / 'config.json').is_file()]
    if not candidates:
        raise FileNotFoundError(f'{dirname} was not found. Attach the Transformer notebook output.')
    return sorted(candidates, key=lambda path: len(str(path)))[0]

model_dir = find_named_directory('best_transformer_encoder', MODEL_DIR_OVERRIDE)
transformer_summary_path = find_named_file('transformer_run_summary.json', TRANSFORMER_SUMMARY_OVERRIDE)
manifest_path = find_named_file('sampled_split_manifest.csv', MANIFEST_OVERRIDE)
baseline_model_path = find_named_file('best_classical_baseline.joblib', BASELINE_MODEL_OVERRIDE, required=False)
transformer_summary = json.loads(transformer_summary_path.read_text(encoding='utf-8'))

if not transformer_summary.get('promoted_to_test'):
    raise RuntimeError('Transformer run was not promoted to test.')
if transformer_summary.get('selected_model') != 'xlm_roberta_base':
    print('Warning: selected model differs from the expected XLM-R run:', transformer_summary.get('selected_model'))

print('Transformer model:', model_dir)
print('Transformer summary:', transformer_summary_path)
print('Split manifest:', manifest_path)
print('Classical baseline:', baseline_model_path or 'not attached; paired comparison will be skipped')


## Recover the locked test split from the pinned source

Only test identifiers in the manifest are read. The source file is verified by size and SHA-256, and text is reconstructed with the exact training-time preprocessing.


In [ ]:
def file_sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def validate_source(path: Path) -> Path:
    if path.stat().st_size != HF_EXPECTED_SIZE:
        raise ValueError(f'Unexpected source size: {path.stat().st_size:,}')
    actual_hash = file_sha256(path)
    if actual_hash != HF_EXPECTED_SHA256:
        raise ValueError(f'Source SHA256 mismatch: {actual_hash}')
    return path

def assert_hf_network():
    try:
        socket.getaddrinfo('huggingface.co', 443, type=socket.SOCK_STREAM)
    except socket.gaierror as error:
        raise RuntimeError('Enable Internet in Kaggle and restart the session.') from error

def direct_download(target: Path) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + '.part')
    request = urllib.request.Request(HF_DOWNLOAD_URL, headers={'User-Agent': 'digikala-evaluation/1.0'})
    with urllib.request.urlopen(request, timeout=120) as response, partial.open('wb') as output:
        while block := response.read(8 * 1024 * 1024):
            output.write(block)
    partial.replace(target)
    return target

def get_source_csv(override: str | None = None) -> Path:
    if override:
        return validate_source(Path(override))
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        candidates = [
            path for path in root.rglob(HF_FILENAME)
            if path.is_file() and path.stat().st_size == HF_EXPECTED_SIZE
        ]
        for candidate in candidates:
            try:
                print('Validating attached source:', candidate)
                return validate_source(candidate)
            except ValueError:
                pass
    assert_hf_network()
    try:
        from huggingface_hub import hf_hub_download
        path = Path(hf_hub_download(
            repo_id=HF_REPO_ID, repo_type='dataset', filename=HF_FILENAME,
            revision=HF_REVISION, cache_dir=str(OUTPUT_DIR / 'hf_cache'),
        ))
        return validate_source(path)
    except Exception as error:
        print(f'huggingface_hub failed ({type(error).__name__}: {error}); using direct URL.')
        return validate_source(direct_download(OUTPUT_DIR / 'hf_data' / HF_FILENAME))

comments_path = get_source_csv(COMMENTS_PATH_OVERRIDE)
print('Verified source:', comments_path)


In [ ]:
TEXT_COLUMNS = ['id', 'title', 'body', 'advantages', 'disadvantages', 'recommendation_status', 'product_id']
NULL_TOKENS = {'', 'nan', 'none', 'null', 'na', 'n/a'}
ARABIC_TO_PERSIAN = str.maketrans({'\u064a': '\u06cc', '\u0649': '\u06cc', '\u0643': '\u06a9'})

def normalize_text_series(series: pd.Series) -> pd.Series:
    out = series.fillna('').astype(str)
    stripped_lower = out.str.strip().str.lower()
    out = out.mask(stripped_lower.isin(NULL_TOKENS), '')
    out = out.str.normalize('NFKC').str.translate(ARABIC_TO_PERSIAN)
    out = out.str.replace('\ufeff', '', regex=False)
    out = out.str.replace(r'\s+', ' ', regex=True).str.strip()
    return out

def build_model_text(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in ['title', 'body', 'advantages', 'disadvantages']:
        frame[column] = normalize_text_series(frame[column])
    parts = []
    for column, tag in [('title', '[TITLE]'), ('body', '[BODY]'), ('advantages', '[ADVANTAGES]'), ('disadvantages', '[DISADVANTAGES]')]:
        parts.append(np.where(frame[column].ne(''), tag + ' ' + frame[column] + ' ', ''))
    full_text = pd.Series(parts[0], index=frame.index)
    for part in parts[1:]:
        full_text = full_text + pd.Series(part, index=frame.index)
    frame['text_full'] = full_text.str.replace(r'\s+', ' ', regex=True).str.strip()
    return frame

manifest = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
required_columns = {'id', 'product_id', 'text_group_id', 'recommendation_status', 'split'}
if not required_columns.issubset(manifest.columns):
    raise ValueError(f'Manifest is missing: {sorted(required_columns - set(manifest.columns))}')
test_manifest = manifest[manifest['split'].eq('test')].copy()
if len(test_manifest) != EXPECTED_TEST_ROWS or test_manifest['text_group_id'].nunique() != EXPECTED_TEST_GROUPS:
    raise RuntimeError('Test manifest does not match the locked baseline split.')
if test_manifest['recommendation_status'].value_counts().to_dict() != EXPECTED_TEST_LABEL_COUNTS:
    raise RuntimeError('Test label profile does not match the locked baseline split.')

wanted_ids = set(test_manifest['id'])
selected_chunks = []
for chunk_number, chunk in enumerate(pd.read_csv(
    comments_path, usecols=TEXT_COLUMNS, dtype=str, chunksize=CHUNK_SIZE,
    keep_default_na=False, na_filter=False, encoding='utf-8-sig',
), start=1):
    chosen = chunk[chunk['id'].isin(wanted_ids)].copy()
    if not chosen.empty:
        selected_chunks.append(chosen)
    if chunk_number % 5 == 0:
        print(f'Chunks: {chunk_number:,} | recovered: {sum(map(len, selected_chunks)):,}/{len(wanted_ids):,}')

test_text = pd.concat(selected_chunks, ignore_index=True).drop_duplicates('id', keep='first')
test_text = build_model_text(test_text)
test_df = test_manifest.merge(
    test_text.drop(columns=['product_id', 'recommendation_status'], errors='ignore'),
    on='id', how='left', validate='one_to_one',
)
if test_df['text_full'].isna().any():
    raise RuntimeError('Some locked test rows were not recovered from the source.')
test_df['label_id'] = test_df['recommendation_status'].map(LABEL2ID).astype(int)
print('Locked test recovered:', len(test_df), 'rows')
display(test_df[['id', 'text_full', 'recommendation_status']].sample(5, random_state=SEED))


## Reproduce predictions and primary metrics

Softmax is stored only as a `model score` and is not a calibrated statistical probability. Reproduced results must match the original run summary.


In [ ]:
inference_config_path = model_dir / 'inference_config.json'
inference_config = json.loads(inference_config_path.read_text(encoding='utf-8')) if inference_config_path.exists() else {}
max_length = int(inference_config.get('max_length', transformer_summary['validation_candidates'][-1]['max_length']))
# This is XLM-R, not Mistral. False preserves the training tokenizer and silences the Transformers 4.57 false-positive warning.
tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True, fix_mistral_regex=False)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
model.eval()

def infer_logits(texts, batch_size=PREDICT_BATCH_SIZE):
    all_logits = []
    started = time.perf_counter()
    for start in range(0, len(texts), batch_size):
        batch_texts = list(texts[start:start + batch_size])
        encoded = tokenizer(
            batch_texts, truncation=True, max_length=max_length,
            padding=True, pad_to_multiple_of=8, return_tensors='pt',
        ).to(device)
        with torch.inference_mode():
            if device.type == 'cuda':
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(**encoded).logits
            else:
                logits = model(**encoded).logits
        all_logits.append(logits.float().cpu().numpy())
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return np.concatenate(all_logits), float(time.perf_counter() - started)

test_logits, reproduction_predict_seconds = infer_logits(test_df['text_full'].tolist())
shifted = test_logits - test_logits.max(axis=1, keepdims=True)
test_scores = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
test_prediction_ids = test_logits.argmax(axis=1)
test_df['prediction_id'] = test_prediction_ids
test_df['prediction'] = [ID2LABEL[index] for index in test_prediction_ids]
test_df['confidence_score'] = test_scores.max(axis=1)
sorted_scores = np.sort(test_scores, axis=1)
test_df['score_margin'] = sorted_scores[:, -1] - sorted_scores[:, -2]
for index, label in ID2LABEL.items():
    test_df[f'score_{label}'] = test_scores[:, index]

def calculate_metrics(y_true, y_pred):
    result = {
        'macro_f1': float(f1_score(y_true, y_pred, labels=list(ID2LABEL), average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, labels=list(ID2LABEL), average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }
    report = classification_report(y_true, y_pred, labels=list(ID2LABEL), output_dict=True, zero_division=0)
    for index, label in ID2LABEL.items():
        result[f'precision_{label}'] = float(report[str(index)]['precision'])
        result[f'recall_{label}'] = float(report[str(index)]['recall'])
        result[f'f1_{label}'] = float(report[str(index)]['f1-score'])
        result[f'support_{label}'] = int(report[str(index)]['support'])
    return result

transformer_metrics = calculate_metrics(test_df['label_id'], test_df['prediction_id'])
reported_macro = float(transformer_summary['test_metrics']['macro_f1'])
reproduction_delta = abs(transformer_metrics['macro_f1'] - reported_macro)
if reproduction_delta > MAX_REPRODUCTION_MACRO_F1_DELTA:
    raise RuntimeError(
        f'Reproduced Macro-F1 differs by {reproduction_delta}, which exceeds '
        f'the allowed cross-run tolerance {MAX_REPRODUCTION_MACRO_F1_DELTA}.'
    )
if reproduction_delta > 1e-6:
    print(
        f'Note: cross-run Macro-F1 delta={reproduction_delta:.6f}; '
        'this is within the locked 0.001 reproducibility tolerance.'
    )

print('Reproduced transformer metrics:')
print(json.dumps(transformer_metrics, ensure_ascii=False, indent=2))
print('Prediction seconds:', round(reproduction_predict_seconds, 3))


In [ ]:
baseline_metrics = None
baseline_prediction_ids = None
if baseline_model_path is not None:
    baseline_bundle = joblib.load(baseline_model_path)
    baseline_model = baseline_bundle['model']
    baseline_text_column = baseline_bundle['text_column']
    baseline_prediction = baseline_model.predict(test_df[baseline_text_column])
    baseline_prediction_ids = pd.Series(baseline_prediction).map(LABEL2ID).to_numpy(dtype=int)
    test_df['baseline_prediction'] = baseline_prediction
    baseline_metrics = calculate_metrics(test_df['label_id'], baseline_prediction_ids)
    if abs(baseline_metrics['macro_f1'] - BASELINE_TEST['macro_f1']) > 1e-9:
        raise RuntimeError('Baseline artifact does not reproduce the recorded baseline test score.')
else:
    print('Baseline model artifact unavailable; using recorded aggregate metrics only.')

comparison_rows = [
    {'model': 'classical_baseline', **(baseline_metrics or BASELINE_TEST)},
    {'model': transformer_summary['selected_model'], **transformer_metrics},
]
comparison = pd.DataFrame(comparison_rows)
display(comparison[['model', 'macro_f1', 'weighted_f1', 'accuracy', 'f1_recommended', 'f1_not_recommended', 'f1_no_idea']])

per_class_report = pd.DataFrame(classification_report(
    test_df['label_id'], test_df['prediction_id'], labels=list(ID2LABEL),
    target_names=VALID_LABELS, output_dict=True, zero_division=0,
)).T.loc[VALID_LABELS].reset_index(names='label')

cm = confusion_matrix(test_df['label_id'], test_df['prediction_id'], labels=list(ID2LABEL))
cm_normalized = cm / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, matrix, title, value_format in [
    (axes[0], cm, 'Confusion matrix — counts', 'd'),
    (axes[1], cm_normalized, 'Confusion matrix — row normalized', '.2f'),
]:
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(len(VALID_LABELS)), VALID_LABELS, rotation=20)
    ax.set_yticks(range(len(VALID_LABELS)), VALID_LABELS)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    threshold = matrix.max() / 2
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = format(matrix[row, column], value_format)
            ax.text(column, row, value, ha='center', va='center', color='white' if matrix[row, column] > threshold else 'black')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
confusion_path = OUTPUT_DIR / 'recommendation_confusion_matrix.png'
plt.savefig(confusion_path, dpi=180, bbox_inches='tight')
plt.show()


## Confidence intervals with group bootstrap

The resampling unit is `text_group_id`, so duplicate-text rows are sampled together. When the baseline artifact is attached, the difference between models also receives a paired confidence interval.


In [ ]:
def confusion_by_group(frame, prediction_ids):
    group_codes, unique_groups = pd.factorize(frame['text_group_id'], sort=True)
    matrices = np.zeros((len(unique_groups), len(VALID_LABELS), len(VALID_LABELS)), dtype=np.int32)
    np.add.at(matrices, (group_codes, frame['label_id'].to_numpy(), prediction_ids), 1)
    return matrices

def macro_f1_from_confusion(matrix):
    true_positive = np.diag(matrix).astype(float)
    false_positive = matrix.sum(axis=0) - true_positive
    false_negative = matrix.sum(axis=1) - true_positive
    denominator = 2 * true_positive + false_positive + false_negative
    class_f1 = np.divide(2 * true_positive, denominator, out=np.zeros_like(true_positive), where=denominator != 0)
    return float(class_f1.mean())

transformer_group_cm = confusion_by_group(test_df, test_prediction_ids)
baseline_group_cm = confusion_by_group(test_df, baseline_prediction_ids) if baseline_prediction_ids is not None else None
group_count = transformer_group_cm.shape[0]
probabilities = np.full(group_count, 1.0 / group_count)
rng = np.random.default_rng(SEED)
transformer_bootstrap = np.empty(BOOTSTRAP_ITERATIONS, dtype=float)
baseline_bootstrap = np.empty(BOOTSTRAP_ITERATIONS, dtype=float) if baseline_group_cm is not None else None

for iteration in range(BOOTSTRAP_ITERATIONS):
    counts = rng.multinomial(group_count, probabilities)
    transformer_matrix = np.tensordot(counts, transformer_group_cm, axes=(0, 0))
    transformer_bootstrap[iteration] = macro_f1_from_confusion(transformer_matrix)
    if baseline_group_cm is not None:
        baseline_matrix = np.tensordot(counts, baseline_group_cm, axes=(0, 0))
        baseline_bootstrap[iteration] = macro_f1_from_confusion(baseline_matrix)

bootstrap_result = {
    'iterations': BOOTSTRAP_ITERATIONS,
    'resampling_unit': 'text_group_id',
    'transformer_macro_f1_ci95': [float(value) for value in np.quantile(transformer_bootstrap, [0.025, 0.975])],
}
if baseline_bootstrap is not None:
    paired_delta = transformer_bootstrap - baseline_bootstrap
    bootstrap_result['baseline_macro_f1_ci95'] = [float(value) for value in np.quantile(baseline_bootstrap, [0.025, 0.975])]
    bootstrap_result['paired_macro_f1_delta_ci95'] = [float(value) for value in np.quantile(paired_delta, [0.025, 0.975])]
    bootstrap_result['paired_probability_transformer_better'] = float(np.mean(paired_delta > 0))

print(json.dumps(bootstrap_result, ensure_ascii=False, indent=2))


## Slice evaluation and failure analysis

Performance is reported by length, presence of text components, and truncation. Failure outputs include a stratified sample for human review; complete its manual columns after downloading the CSV.


In [ ]:
token_lengths = []
for start in range(0, len(test_df), 512):
    encoded = tokenizer(
        test_df['text_full'].iloc[start:start + 512].tolist(),
        truncation=False, padding=False, return_length=True, verbose=False,
    )
    token_lengths.extend(encoded['length'])
test_df['token_length'] = np.asarray(token_lengths, dtype=int)
test_df['char_length'] = test_df['text_full'].str.len().astype(int)
test_df['was_truncated'] = test_df['token_length'] > max_length
test_df['has_title'] = test_df['title'].ne('')
test_df['has_advantages'] = test_df['advantages'].ne('')
test_df['has_disadvantages'] = test_df['disadvantages'].ne('')
test_df['has_structured_pros_or_cons'] = test_df['has_advantages'] | test_df['has_disadvantages']
test_df['length_bucket'] = pd.cut(
    test_df['token_length'], bins=[0, 16, 48, 96, max_length, np.inf],
    labels=['very_short_1_16', 'short_17_48', 'medium_49_96', f'long_97_{max_length}', f'truncated_gt_{max_length}'],
    include_lowest=True,
).astype(str)

slice_masks = {
    'all_test': np.ones(len(test_df), dtype=bool),
    'title_missing': ~test_df['has_title'],
    'title_present': test_df['has_title'],
    'structured_pros_or_cons_present': test_df['has_structured_pros_or_cons'],
    'structured_pros_or_cons_missing': ~test_df['has_structured_pros_or_cons'],
    'truncated': test_df['was_truncated'],
    'not_truncated': ~test_df['was_truncated'],
}
for bucket in test_df['length_bucket'].dropna().unique():
    slice_masks[f'length::{bucket}'] = test_df['length_bucket'].eq(bucket)

slice_rows = []
for slice_name, mask in slice_masks.items():
    subset = test_df.loc[np.asarray(mask)]
    if subset.empty:
        continue
    metrics = calculate_metrics(subset['label_id'], subset['prediction_id'])
    slice_rows.append({
        'slice': slice_name,
        'rows': int(len(subset)),
        'groups': int(subset['text_group_id'].nunique()),
        **{f'true_{label}': int(subset['recommendation_status'].eq(label).sum()) for label in VALID_LABELS},
        **metrics,
    })
slice_results = pd.DataFrame(slice_rows).sort_values(['rows', 'slice'], ascending=[False, True])
display(slice_results[['slice', 'rows', 'macro_f1', 'f1_recommended', 'f1_not_recommended', 'f1_no_idea']])

errors = test_df[test_df['label_id'].ne(test_df['prediction_id'])].copy()
errors['confusion_pair'] = errors['recommendation_status'] + ' -> ' + errors['prediction']
errors['error_confidence_band'] = pd.cut(
    errors['confidence_score'], bins=[0, 0.55, 0.80, 1.0],
    labels=['low_le_0.55', 'medium_0.55_0.80', 'high_ge_0.80'], include_lowest=True,
).astype(str)
display(errors.groupby(['confusion_pair', 'error_confidence_band'], observed=True).size().unstack(fill_value=0))

review_parts = []
for pair, group in errors.groupby('confusion_pair'):
    review_parts.append(group.sample(min(10, len(group)), random_state=SEED))
manual_review = pd.concat(review_parts, ignore_index=True)
manual_review['manual_failure_type'] = ''
manual_review['reviewer_notes'] = ''
manual_review['label_suspected_noisy'] = ''


## Latency, throughput, and cost

Timing includes tokenization, tensor transfer, and the forward pass. Batch 1 represents online requests and batch 32 represents bulk processing. A 250 ms p95 budget is an initial, adjustable engineering gate.


In [ ]:
def synchronize_device():
    if device.type == 'cuda':
        torch.cuda.synchronize()

def timed_inference_batch(texts):
    synchronize_device()
    started = time.perf_counter()
    encoded = tokenizer(
        list(texts), truncation=True, max_length=max_length,
        padding=True, pad_to_multiple_of=8, return_tensors='pt',
    ).to(device)
    with torch.inference_mode():
        if device.type == 'cuda':
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                _ = model(**encoded).logits
        else:
            _ = model(**encoded).logits
    synchronize_device()
    return (time.perf_counter() - started) * 1000

latency_sample = test_df.sample(
    min(max(LATENCY_SINGLE_SAMPLES, LATENCY_BATCH_SAMPLES), len(test_df)), random_state=SEED
)['text_full'].tolist()
for _ in range(10):
    timed_inference_batch(latency_sample[:1])

if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
single_latencies = np.array([
    timed_inference_batch([text]) for text in latency_sample[:LATENCY_SINGLE_SAMPLES]
])

batch_latencies = []
batch_rows = 0
batch_total_started = time.perf_counter()
for start in range(0, min(LATENCY_BATCH_SAMPLES, len(latency_sample)), LATENCY_BATCH_SIZE):
    texts = latency_sample[start:start + LATENCY_BATCH_SIZE]
    if not texts:
        break
    batch_latencies.append(timed_inference_batch(texts))
    batch_rows += len(texts)
batch_total_seconds = time.perf_counter() - batch_total_started
batch_latencies = np.asarray(batch_latencies)

model_size_bytes = sum(path.stat().st_size for path in model_dir.rglob('*') if path.is_file())
candidate_training_seconds = sum(float(row['fit_seconds']) for row in transformer_summary['validation_candidates'])
final_training_seconds = float(transformer_summary.get('final_fit_seconds') or 0.0)
latency_results = {
    'device': str(device),
    'gpu': gpu_name,
    'max_length': max_length,
    'single_request_samples': int(len(single_latencies)),
    'single_request_latency_ms': {
        'mean': float(single_latencies.mean()),
        'p50': float(np.quantile(single_latencies, 0.50)),
        'p95': float(np.quantile(single_latencies, 0.95)),
        'p99': float(np.quantile(single_latencies, 0.99)),
    },
    'batch_size': LATENCY_BATCH_SIZE,
    'batch_count': int(len(batch_latencies)),
    'batch_latency_ms': {
        'mean': float(batch_latencies.mean()),
        'p50': float(np.quantile(batch_latencies, 0.50)),
        'p95': float(np.quantile(batch_latencies, 0.95)),
    },
    'batch_throughput_rows_per_second': float(batch_rows / batch_total_seconds),
    'peak_gpu_memory_bytes': int(torch.cuda.max_memory_allocated()) if device.type == 'cuda' else None,
    'model_artifact_size_bytes': int(model_size_bytes),
    'api_requests': 0,
    'api_tokens': 0,
    'api_cost_usd': 0.0,
    'candidate_plus_final_training_gpu_hours': float((candidate_training_seconds + final_training_seconds) / 3600),
}
print(json.dumps(latency_results, ensure_ascii=False, indent=2))


## Release gates, integration contract, and saved outputs

Gates remain fixed before future experiments. The integration contract's `source` distinguishes an observed label from a model prediction; an LLM must never present a prediction as observed fact.


In [ ]:
macro_gain = transformer_metrics['macro_f1'] - BASELINE_TEST['macro_f1']
paired_ci = bootstrap_result.get('paired_macro_f1_delta_ci95')
release_gates = {
    'metric_reproduction_within_0.001': reproduction_delta <= MAX_REPRODUCTION_MACRO_F1_DELTA,
    'test_macro_f1_gain_at_least_0.02': macro_gain >= MIN_TEST_MACRO_GAIN,
    'no_idea_f1_improves': transformer_metrics['f1_no_idea'] > BASELINE_TEST['f1_no_idea'],
    'recommended_f1_non_regression': transformer_metrics['f1_recommended'] >= BASELINE_TEST['f1_recommended'] - MAX_PER_CLASS_F1_REGRESSION,
    'not_recommended_f1_non_regression': transformer_metrics['f1_not_recommended'] >= BASELINE_TEST['f1_not_recommended'] - MAX_PER_CLASS_F1_REGRESSION,
    'single_request_p95_within_budget': latency_results['single_request_latency_ms']['p95'] <= P95_SINGLE_LATENCY_BUDGET_MS,
}
if paired_ci is not None:
    release_gates['paired_bootstrap_delta_ci_lower_above_zero'] = paired_ci[0] > 0
release_decision = 'PASS' if all(release_gates.values()) else 'FAIL'

integration_contract = {
    'component': 'recommendation_prediction',
    'version': '1.0.0',
    'input_schema': {
        'comment_id': 'string|null',
        'product_id': 'string|null',
        'title': 'string',
        'body': 'string',
        'advantages': 'string',
        'disadvantages': 'string',
    },
    'output_schema': {
        'label': VALID_LABELS,
        'scores': {label: 'float_uncalibrated' for label in VALID_LABELS},
        'model_version': transformer_summary['selected_resolved_revision'],
        'preprocessing_version': 'fa_light_v1',
        'source': ['observed', 'model_prediction'],
        'latency_ms': 'float',
    },
    'integration_rule': 'Use observed recommendation_status when present; predict only missing or new comments.',
}

predictions_path = OUTPUT_DIR / 'recommendation_test_predictions.csv'
per_class_path = OUTPUT_DIR / 'recommendation_per_class.csv'
slices_path = OUTPUT_DIR / 'recommendation_slice_results.csv'
failures_path = OUTPUT_DIR / 'recommendation_failure_cases.csv'
review_path = OUTPUT_DIR / 'recommendation_manual_review_sample.csv'
latency_path = OUTPUT_DIR / 'recommendation_latency_results.json'
contract_path = OUTPUT_DIR / 'recommendation_integration_contract.json'
summary_path = OUTPUT_DIR / 'recommendation_evaluation_summary.json'
release_card_path = OUTPUT_DIR / 'recommendation_release_card.md'

prediction_columns = [
    'id', 'product_id', 'text_group_id', 'recommendation_status', 'prediction',
    'confidence_score', 'score_margin', *[f'score_{label}' for label in VALID_LABELS],
    'token_length', 'was_truncated', 'length_bucket',
]
if 'baseline_prediction' in test_df:
    prediction_columns.append('baseline_prediction')
test_df[prediction_columns].to_csv(predictions_path, index=False)
per_class_report.to_csv(per_class_path, index=False)
slice_results.to_csv(slices_path, index=False)
errors.to_csv(failures_path, index=False)
manual_review.to_csv(review_path, index=False)
latency_path.write_text(json.dumps(latency_results, ensure_ascii=False, indent=2), encoding='utf-8')
contract_path.write_text(json.dumps(integration_contract, ensure_ascii=False, indent=2), encoding='utf-8')

evaluation_summary = {
    'task': 'digikala_recommendation_status_final_evaluation',
    'decision': release_decision,
    'release_gates': release_gates,
    'seed': SEED,
    'source_sha256': HF_EXPECTED_SHA256,
    'manifest': str(manifest_path),
    'test_rows': int(len(test_df)),
    'test_groups': int(test_df['text_group_id'].nunique()),
    'model': transformer_summary['selected_model'],
    'checkpoint': transformer_summary['selected_checkpoint'],
    'resolved_revision': transformer_summary['selected_resolved_revision'],
    'max_length': max_length,
    'baseline_metrics': baseline_metrics or BASELINE_TEST,
    'transformer_metrics': transformer_metrics,
    'absolute_macro_f1_gain': float(macro_gain),
    'metric_reproduction_delta': float(reproduction_delta),
    'metric_reproduction_tolerance': MAX_REPRODUCTION_MACRO_F1_DELTA,
    'bootstrap': bootstrap_result,
    'latency_and_cost': latency_results,
    'slice_count': int(len(slice_results)),
    'error_rows': int(len(errors)),
    'manual_review_rows': int(len(manual_review)),
    'confidence_note': 'Softmax outputs are uncalibrated model scores.',
    'versions': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'platform': platform.platform(),
        'gpu': gpu_name,
    },
}
summary_path.write_text(json.dumps(evaluation_summary, ensure_ascii=False, indent=2), encoding='utf-8')

paired_text = (
    f"Paired group-bootstrap delta CI95: [{paired_ci[0]:.4f}, {paired_ci[1]:.4f}]"
    if paired_ci is not None else 'Paired comparison unavailable because the baseline artifact was not attached.'
)
release_card = f"""# Recommendation Prediction Release Card

Decision: **{release_decision}**

## Claim
On the locked group-disjoint Digikala test split, the selected encoder improves three-class recommendation prediction over the classical baseline without a material per-class regression.

## Evidence
- Baseline test Macro-F1: {BASELINE_TEST['macro_f1']:.4f}
- Transformer test Macro-F1: {transformer_metrics['macro_f1']:.4f}
- Absolute gain: {macro_gain:+.4f}
- Transformer no_idea F1: {transformer_metrics['f1_no_idea']:.4f}
- {paired_text}
- Single-request p95 latency: {latency_results['single_request_latency_ms']['p95']:.2f} ms on {gpu_name or 'CPU'}
- API cost: $0; API tokens: 0

## Gates
{json.dumps(release_gates, ensure_ascii=False, indent=2)}

## Known limitation
The no_idea class remains the weakest class and softmax scores are not calibrated probabilities. Test results must not be used for further tuning.
"""
release_card_path.write_text(release_card, encoding='utf-8')

print('Saved artifacts:')
for path in [predictions_path, per_class_path, slices_path, failures_path, review_path, latency_path, contract_path, summary_path, confusion_path, release_card_path]:
    print(f' - {path} ({path.stat().st_size / 1_000_000:.2f} MB)')
print('\n' + '#' * 28 + ' COPY THIS SUMMARY ' + '#' * 28)
print(json.dumps(evaluation_summary, ensure_ascii=False, indent=2))


## Inference function for the integration team

This function demonstrates the component contract. In the final package, preprocessing and model loading should live in an independent module.


In [ ]:
def predict_recommendation(title='', body='', advantages='', disadvantages=''):
    frame = build_model_text(pd.DataFrame([{
        'title': title, 'body': body,
        'advantages': advantages, 'disadvantages': disadvantages,
    }]))
    started = time.perf_counter()
    logits, _ = infer_logits(frame['text_full'].tolist(), batch_size=1)
    elapsed_ms = (time.perf_counter() - started) * 1000
    shifted = logits[0] - logits[0].max()
    scores = np.exp(shifted) / np.exp(shifted).sum()
    prediction_id = int(scores.argmax())
    return {
        'label': ID2LABEL[prediction_id],
        'scores': {ID2LABEL[index]: float(scores[index]) for index in ID2LABEL},
        'scores_are_calibrated_probabilities': False,
        'model_version': transformer_summary['selected_resolved_revision'],
        'preprocessing_version': 'fa_light_v1',
        'source': 'model_prediction',
        'latency_ms': float(elapsed_ms),
    }

examples = [
    {'title': 'excellent', 'body': 'excellent quality; I would buy it again'},
    {'title': 'do not buy', 'body': 'the quality was poor and I returned it'},
    {'title': 'average', 'body': 'reasonable for the price, but I expected more'},
]
display(pd.DataFrame([{**example, **predict_recommendation(**example)} for example in examples]))


## Outputs to retain

Retain the complete `COPY THIS SUMMARY` block and download `recommendation_manual_review_sample.csv`; its samples require human review to complete failure analysis. Preserve all other artifacts with Kaggle `Save Version`.
